# Interactive Pipeline Testbed

In [ ]:
import numpy as np
import pandas as pd
from src import config
from src import data_ingestion
from src import data_understanding
from src import data_preparation
from src import modeling
from src import evaluation
import sys
sys.path.append(str(config.PROJECT_ROOT / ".github" / "scripts"))
import sync_assets
from src import report
from IPython.display import Image, Markdown, display

## 0. Parameter Eksperimen

In [ ]:
# Parameter Eksperimen
TICKER = "BBCA.JK"
RANGE_DAYS = 1825
END_DATE = "auto"
TARGET_COL = "close"
TRAIN_SIZE = 0.90
RANDOM_STATE = 42

# Hyperparameters Grid
TIME_STEPS = [10]
OPTIMIZERS = ["Adam"]
BATCH_SIZES = [8]
LEARNING_RATES = [0.01]
EPOCHS = 50

## 1. Data Ingestion

In [ ]:
df_raw = data_ingestion.run_data_ingestion(
    ticker=TICKER,
    range_days=RANGE_DAYS,
    end_date=END_DATE,
)

display(df_raw.head())

## 2. Data Understanding

In [ ]:
und_summary = data_understanding.run_data_understanding(
    ticker=TICKER,
    target_col=TARGET_COL,
)

print("Ringkasan Data Understanding:")
print(f"- Periode Observasi : {und_summary['min_date']} s/d {und_summary['max_date']}")
print(f"- Total Baris Data  : {und_summary['total_rows']}")
print(f"- Gap Tanggal Bursa : {und_summary['gap_count']}")

In [ ]:
plot_hist_path = config.get_plot_path(TICKER, "historical_price")
display(Image(filename=str(plot_hist_path)))

## 3. Data Preparation

In [ ]:
data_preparation.run_data_preparation(
    ticker=TICKER,
    train_size=TRAIN_SIZE,
    target_col=TARGET_COL,
)

prep_path = config.get_prepared_data_path(TICKER)
prep_data = np.load(prep_path, allow_pickle=True)

print(f"Dataset siap latih tersimpan di: {prep_path.name}")
print(f"- Dimensi Data Latih (Scaled): {prep_data['train_scaled'].shape}")
print(f"- Dimensi Data Uji (Scaled)  : {prep_data['test_scaled'].shape}")
print(f"- Rentang Tanggal Latih      : {prep_data['dates_train'][0]} s/d {prep_data['dates_train'][-1]}")
print(f"- Rentang Tanggal Uji        : {prep_data['dates_test'][0]} s/d {prep_data['dates_test'][-1]}")

## 4.Modeling

In [ ]:
modeling.run_modeling(
    ticker=TICKER,
    time_steps=TIME_STEPS,
    optimizers_list=OPTIMIZERS,
    batch_sizes=BATCH_SIZES,
    learning_rates=LEARNING_RATES,
    models_list=config.MODELS,
    epochs=EPOCHS,
    random_state=RANDOM_STATE,
    target_col=TARGET_COL,
)

# Tampilkan tabel hasil hyperparameter grid search
hp_path = config.get_hyperparameter_path(TICKER)
hp_df = pd.read_csv(hp_path)
display(hp_df)

## 5. Evaluation

In [ ]:
evaluation.run_evaluation(
    ticker=TICKER,
    target_col=TARGET_COL,
)

# Tampilkan plot inferensi aktual vs prediksi untuk masing-masing model
for m in config.MODELS:
    inf_plot = config.get_plot_path(TICKER, f"{m}_inference")
    if inf_plot.exists():
        display(Image(filename=str(inf_plot)))

## 6. Reporting

In [ ]:
# Promosi model champion & sinkronisasi aset backend
sync_assets.promote_champion_models(ticker=TICKER)
sync_assets.sync_assets(ticker=TICKER)

# Pembuatan laporan ringkasan markdown
report_file = config.ARTIFACT_DIR / "report.md"
report_md = report.generate_report(output_path=report_file)

display(Markdown(report_md))